# 실습 환경 설정하기

In [68]:
# 원시 데이터 조회


import pymysql
import pandas as pd


# # 1. DB 연걸



conn = pymysql.connect(
    host='localhost', 
    user='taesub', 
    password='0204',
    database='processsensordb',
    charset='utf8mb4' # 문자열을 UTF-8로 인코딩하여 저장하기 위해 설정
    )




query = """

select 
*
from sensormeasurement sm 
join sensor s 
on sm.sensor_id = s.sensor_id;

"""

df = pd.read_sql(query, conn)
print(df.head())




   measurement_id  run_id  sensor_id         measured_at  measured_value  \
0               1       1          1 2024-03-15 09:05:00           398.0   
1               2       1          1 2024-03-15 09:15:00           402.0   
2               3       1          1 2024-03-15 09:25:00           405.0   
3              15       4          1 2024-03-15 11:15:00           399.0   
4              16       4          1 2024-03-15 11:30:00           401.0   

   sensor_id  equipment_id   sensor_name unit  normal_min  normal_max  
0          1           101  chamber_temp    C       390.0       410.0  
1          1           101  chamber_temp    C       390.0       410.0  
2          1           101  chamber_temp    C       390.0       410.0  
3          1           101  chamber_temp    C       390.0       410.0  
4          1           101  chamber_temp    C       390.0       410.0  


C:\Users\user\AppData\Local\Temp\ipykernel_11628\2426338841.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [135]:
#2
query =""" 
select  RH.run_id, 
E.equipment_id, 
E.model_name, 
PS.step_name, 
PS.process_group, 
RH.run_status, 
RH.start_time, 
RH.lot_id
from RunHistory as RH
Join Equipment as E on RH.equipment_id = E.equipment_id
Join ProcessStep as PS on PS.step_id = RH.step_id
"""
df = pd.read_sql(query, conn)
print(df.head())

   run_id  equipment_id model_name step_name process_group run_status  \
0       1           101   CVD-B200    산화막 증착    Deposition  completed   
1       4           101   CVD-B200    산화막 증착    Deposition  completed   
2       7           101   CVD-B200    산화막 증착    Deposition    warning   
3       2           102  ETCH-A100     식각 공정          Etch  completed   
4       5           102  ETCH-A100     식각 공정          Etch  completed   

           start_time lot_id  
0 2024-03-15 09:00:00   L001  
1 2024-03-15 11:10:00   L002  
2 2024-03-15 14:00:00   L003  
3 2024-03-15 10:00:00   L001  
4 2024-03-15 12:10:00   L002  


C:\Users\user\AppData\Local\Temp\ipykernel_11628\1757587648.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [55]:
# 원시 데이터 조회
df = pd.read_sql(query, conn)
print(df.head())


   measurement_id  run_id  sensor_id         measured_at  measured_value  \
0               1       1          1 2024-03-15 09:05:00           398.0   
1               2       1          1 2024-03-15 09:15:00           402.0   
2               3       1          1 2024-03-15 09:25:00           405.0   
3              15       4          1 2024-03-15 11:15:00           399.0   
4              16       4          1 2024-03-15 11:30:00           401.0   

   sensor_id  equipment_id   sensor_name unit  normal_min  normal_max  
0          1           101  chamber_temp    C       390.0       410.0  
1          1           101  chamber_temp    C       390.0       410.0  
2          1           101  chamber_temp    C       390.0       410.0  
3          1           101  chamber_temp    C       390.0       410.0  
4          1           101  chamber_temp    C       390.0       410.0  


C:\Users\user\AppData\Local\Temp\ipykernel_11628\864206788.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [13]:
# 필요한 데이터 선택
df = df[['run_id', 'sensor_name', 'measured_value']]

print(df.head())

   run_id   sensor_name  measured_value
0       1  chamber_temp           398.0
1       1  chamber_temp           402.0
2       1  chamber_temp           405.0
3       4  chamber_temp           399.0
4       4  chamber_temp           401.0


In [14]:
# 필요한 데이터 선택
temp_df = df[df['sensor_name'] == 'chamber_temp']

print(temp_df)

   run_id   sensor_name  measured_value
0       1  chamber_temp           398.0
1       1  chamber_temp           402.0
2       1  chamber_temp           405.0
3       4  chamber_temp           399.0
4       4  chamber_temp           401.0
5       7  chamber_temp           408.0
6       7  chamber_temp           416.0
7       7  chamber_temp           418.0


In [15]:
# 조건 필터링
high_temp_df = df[df['measured_value'] >= 410]
print(high_temp_df)

    run_id   sensor_name  measured_value
6        7  chamber_temp           416.0
7        7  chamber_temp           418.0
16       2      rf_power           500.0
17       2      rf_power           508.0
18       5      rf_power           495.0
19       5      rf_power           505.0
20       8      rf_power           525.0
21       8      rf_power           530.0


In [20]:
# 데이터 묶기
#예시 센서별 평균 측정값 계산

sensor_mean = df.groupby('sensor_name')['measured_value'].mean()

print(sensor_mean)




sensor_name
chamber_pressure      2.037500
chamber_temp        405.875000
chemical_temp        24.750000
cleaning_flow        20.500000
defect_count          3.500000
gas_flow             97.166667
inspection_temp      25.500000
rf_power            510.500000
Name: measured_value, dtype: float64


In [21]:
print(df.columns)

Index(['run_id', 'sensor_name', 'measured_value'], dtype='str')


In [24]:
# 데이터 묶기
runcount_per_equip = df.groupby('equipment_id')['run_id'].count()
print(runcount_per_equip)

equipment_id
101    16
102    12
103     4
104     4
Name: run_id, dtype: int64


In [25]:
# 5 데이터 형태 변환하기(pivot_table)
pivot_df = df.pivot_table(
    index = 'run_id',
    columns = 'sensor_name',
    values='measured_value'


)

pivot_df

sensor_name,chamber_pressure,chamber_temp,chemical_temp,cleaning_flow,defect_count,gas_flow,inspection_temp,rf_power
run_id,,,,,,,,
1,2.0,401.666667,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,101.0,NaN,504.0
3,NaN,NaN,24.75,20.5,NaN,NaN,NaN,NaN
4,2.0,400.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,99.5,NaN,500.0
6,NaN,NaN,NaN,NaN,3.5,NaN,25.5,NaN
7,2.1,414.000000,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,91.0,NaN,527.5


In [26]:
# 5 데이터 형태 변환하기(pivot_table)

df.pivot_table(
    index = 'run_id',
    columns='sensor_name',
    values='measured_value',
    aggfunc='max'

)

pivot_df

sensor_name,chamber_pressure,chamber_temp,chemical_temp,cleaning_flow,defect_count,gas_flow,inspection_temp,rf_power
run_id,,,,,,,,
1,2.0,401.666667,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,101.0,NaN,504.0
3,NaN,NaN,24.75,20.5,NaN,NaN,NaN,NaN
4,2.0,400.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,99.5,NaN,500.0
6,NaN,NaN,NaN,NaN,3.5,NaN,25.5,NaN
7,2.1,414.000000,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,91.0,NaN,527.5


In [27]:
# 5 데이터 형태 변환하기(pivot_table)

df.pivot_table(
    index = 'run_id',
    columns='sensor_name',
    values='measured_value',
    aggfunc='count'

)

pivot_df

sensor_name,chamber_pressure,chamber_temp,chemical_temp,cleaning_flow,defect_count,gas_flow,inspection_temp,rf_power
run_id,,,,,,,,
1,2.0,401.666667,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,101.0,NaN,504.0
3,NaN,NaN,24.75,20.5,NaN,NaN,NaN,NaN
4,2.0,400.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,99.5,NaN,500.0
6,NaN,NaN,NaN,NaN,3.5,NaN,25.5,NaN
7,2.1,414.000000,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,91.0,NaN,527.5


In [101]:
# 분석용 데이터셋 저장


pivot_df.to_csv(
    'clean_sensor_data.csv',
    encoding = 'utf-8-sig'

)


# 데이터 정재 실습 문제

문제 1 
- RunHistory, Equipment, ProcessStep 테이블을 JOIN하여 공정 실행 이력 데이터를 조회하시오.
- 조회 항목:
-run_id
- equipment_id
- model_name
- step_name
- process_group
-run_status
- start_time
- 조회 결과를 DataFrame으로 저장하고 앞 5개 행을 출력하시오

In [124]:



query = """
select 
r.run_id,
e.equipment_id,
e.model_name,
p.step_name,
p.process_group,
r.run_status,
r.start_time
from runhistory as r
join equipment as e 
on r.equipment_id = e.equipment_id
join processstep as p 
on r.step_id = p.step_id;
"""

df = pd.read_sql(query, conn)
print(df.head())



   run_id  equipment_id model_name step_name process_group run_status  \
0       1           101   CVD-B200    산화막 증착    Deposition  completed   
1       4           101   CVD-B200    산화막 증착    Deposition  completed   
2       7           101   CVD-B200    산화막 증착    Deposition    warning   
3       2           102  ETCH-A100     식각 공정          Etch  completed   
4       5           102  ETCH-A100     식각 공정          Etch  completed   

           start_time  
0 2024-03-15 09:00:00  
1 2024-03-15 11:10:00  
2 2024-03-15 14:00:00  
3 2024-03-15 10:00:00  
4 2024-03-15 12:10:00  


C:\Users\user\AppData\Local\Temp\ipykernel_11628\115073645.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


# ② 필요한 데이터 선택
문제 2 
- 현재 DataFrame에서 다음 컬럼만 선택하여 새로운 DataFrame을 생성하시오.
- run_id, step_name, run_status
- 결과를 출력하시오.

In [123]:
 #인덱스 자체를 리스트 형태로 넣으면 됨...

print(df[['run_id', 'step_name', 'run_status']])

KeyError: "['run_id', 'run_status'] not in index"

문제 3 
- 현재 DataFrame에서 다음 컬럼만 선택하여 새로운 DataFrame을 생성하시오.
- equipment_id, model_name, process_group
- 결과를 출력하시오.

In [ ]:


temp_df = df[['equipment_id', 'model_name', 'process_group']]

temp_df

KeyError: "['equipment_id', 'model_name'] not in index"

# ③ 조건에 맞게 정리
문제 4 
- 공정 상태(run_status)가 ‘warning’인 데이터만 추출하시오.
- 결과를 새로운 DataFrame으로 저장하시오

In [79]:


newdf = df[df['run_status'] == 'warning']
newdf

,run_id,equipment_id,model_name,step_name,process_group,run_status,start_time
2,7,101,CVD-B200,산화막 증착,Deposition,warning,2024-03-15 14:00:00
5,8,102,ETCH-A100,식각 공정,Etch,warning,2024-03-15 15:00:00


문제 5
- 공정 그룹(process_group)이 'ETCH'인 데이터만 조회하시오.
- 조회 결과를 출력하시오.

In [131]:
# query = """

# select 
# *
# from processstep
# where process_group = 'Etch'
# ;
# """


# df = pd.read_sql(query, conn)

# df


temp_df = df[df['process_group'] == 'Etch']

print(temp_df)



   step_id step_name process_group
0        2     식각 공정          Etch


# ④ 데이터 묶기 (groupby)
문제 6 
- 공정 그룹별 공정 실행 횟수를 계산하시오. 출력 결과에는 다음 정보가 포함되어야 한다.
- process_group
-공정 실행 횟수

In [80]:

newdf = df.groupby('process_group')['run_id'].count()

newdf

process_group
Cleaning      1
Deposition    3
Etch          3
Inspection    1
Name: run_id, dtype: int64

문제 7
- 장비별 수행 공정 수를 계산하시오. 출력 결과에는 다음 정보가 포함되어야 한다.
- equipment_id
- model_name
-수행 공정 수

In [139]:
newdf1 = df.groupby(['equipment_id', 'model_name'])['run_id'].count()
newdf1

newdf1 = newdf1.reset_index()

print(newdf1)
print(type(newdf1))

newdf1.columns

newdf1.columns = ['equipment_id', 'model_name', 'run_count']  
print(newdf1)  

   equipment_id    model_name  run_id
0           101      CVD-B200       3
1           102     ETCH-A100       3
2           103    CLEAN-D300       1
3           104  INSPECT-E500       1
<class 'pandas.DataFrame'>
   equipment_id    model_name  run_count
0           101      CVD-B200          3
1           102     ETCH-A100          3
2           103    CLEAN-D300          1
3           104  INSPECT-E500          1


# ⑤ 형태 변환 (pivot)
문제 8 다음 조건으로 pivot_table을 생성하고 결과를 출력하시오.
-index: equipment_id
- columns: process_group
- values: run_id
- aggfunc: count

In [ ]:
pivot_df = df.pivot_table(
    index = 'equipment_id',
    columns = 'process_group',
    values = 'run_id',
    aggfunc = 'count'
)


pivot_df

process_group
Cleaning      NaN
Deposition    3.0
Etch          NaN
Inspection    NaN
Name: 101, dtype: float64

In [102]:
pivot_df.loc[101]

process_group
Cleaning      NaN
Deposition    3.0
Etch          NaN
Inspection    NaN
Name: 101, dtype: float64

문제 9 
다음 조건으로 pivot_table을 생성하고 결과를 출력하시오.
- index: lot_id
- columns: run_status
- values: run_id
- aggfunc: count

In [141]:
pivot_df = df.pivot_table(
    index = 'lot_id',
    columns = 'run_status',
    values = 'run_id',
    aggfunc = 'count'
)


pivot_df

run_status,completed,warning
lot_id,,
L001,3.0,NaN
L002,3.0,NaN
L003,NaN,2.0


# 분석용 데이터셋 생성
문제 10 
- 다음 조건에 맞는 공정 실행 분석용 데이터셋을 생성하시오.
- equipment_id 기준으로 데이터 구성
- process_group을 컬럼으로 변환
-수행 횟수를 값으로 사용
-결과를 CSV 파일로 저장
• 파일 이름: equipment_process_summary.csv
• UTF-8 인코딩 적용

In [142]:
pivot_df = pivot_df.reset_index()


pivot_df.to_csv(
    'temp_df.csv',
    index = False,
    encoding = 'utf-8-sig'
)